In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, recall_score, roc_auc_score
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
import pandas as pd
import numpy as np

In [ ]:
def categorize_conflict(x, q1, q3):
    if x == 0:
        return 0
    elif x <= q1:
        return 1
    elif x <= q3:
        return 2
    else:
        return 3

In [ ]:
parquet_path = '../data/output/grid_conflict_climate_2019_23.parquet'

#Drop nas #
df = pd.read_parquet(parquet_path)
df = df.dropna()
df['target'] = (df['conflict_count'] >= 1).astype(int)
features = df.drop(['GEOID', 'conflict_count', 'target'], axis=1)
features = pd.get_dummies(features, columns=['year'], prefix='year')
X = features
y = df['target']

In [ ]:
# parquet_path = '../data/output/grid_conflict_climate_2019_23.parquet'

# #Drop nas #
# df = pd.read_parquet(parquet_path)
# df = df.dropna()
# new_df = df[df["conflict_count"] > 0]
# q1, q2, q3 = new_df['conflict_count'].quantile(0.25), new_df['conflict_count'].quantile(0.50), new_df['conflict_count'].quantile(0.75)

# df['target'] = df['conflict_count'].apply(lambda x: categorize_conflict(x, q1, q3))
# features = df.drop(['GEOID', 'conflict_count', 'target'], axis=1)
# features = pd.get_dummies(features, columns=['year'], prefix='year')
# X = features
# y = df['target']

In [ ]:
# Preprocess data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train with gradient descent
# 'saga' is a variant of stochastic gradient descent

param_grid = {
    'estimator__C': np.logspace(-3, 2, 6),           # C: [0.001, 0.01, 0.1, 1, 10, 100]
    'estimator__l1_ratio': np.linspace(0, 1, 5)      # l1_ratio: [0, 0.25, 0.5, 0.75, 1]
}

base_model = LogisticRegression(
    solver='saga',
    penalty='elasticnet',
    max_iter=1000,
    random_state=42
)

ovr_model = OneVsRestClassifier(base_model)

grid = GridSearchCV(
    ovr_model,
    param_grid,
    cv=5,
    scoring='recall_weighted',  # or another multilabel-compatible metric
    verbose=1,
    n_jobs=-1
)

grid.fit(X_train, y_train)

# Results
print("Best C:", grid.best_params_['estimator__C'])
print("Best l1_ratio:", grid.best_params_['estimator__l1_ratio'])
print("Best score:", grid.best_score_)

### Class imbalance

In [ ]:
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report, recall_score

# Use the same preprocessing steps as before
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE for class imbalance
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# Define the cross-validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define parameter grid to search
param_grid = {
    'n_estimators': [100, 200, 300, 500],        # Number of trees
    'max_depth': [None, 10, 20],       # Maximum depth of trees
    'min_samples_split': [2, 5, 10],   # Minimum samples required to split
    'min_samples_leaf': [1, 2, 4],     # Minimum samples required at leaf node
    'class_weight': ['balanced', None] # Adjust weights inversely proportional to class frequencies
}

# Initialize the Random Forest classifier
rf = RandomForestClassifier(random_state=42)

# Set up GridSearchCV with recall as the scoring metric
# You can use 'f1' or 'recall' depending on your priority
grid_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=50,                    # Try 50 random combinations
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='recall',     # Optimize for recall
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Fit the model
print("Training Random Forest with GridSearchCV...")
grid_search.fit(X_train_smote, y_train_smote)

# Get best model
best_rf = grid_search.best_estimator_
print(f"Best parameters: {grid_search.best_params_}")

# Predict on test set
y_pred_rf = best_rf.predict(X_test_scaled)

# Evaluate
print("\n--- Random Forest Model (Optimized) ---")
print(f"Accuracy: {best_rf.score(X_test_scaled, y_test):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf, pos_label=1):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_rf, pos_label=1):.4f}")

from sklearn.metrics import precision_score
print(f"Precision: {precision_score(y_test, y_pred_rf, pos_label=1):.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred_rf)
print(cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Conflict (0)', 'Conflict (1)'],
            yticklabels=['No Conflict (0)', 'Conflict (1)'])
plt.xlabel('Predicted Label', fontsize=14)
plt.ylabel('True Label', fontsize=14)
plt.title('Confusion Matrix - Random Forest', fontsize=16)
plt.tight_layout()
plt.show()

# Feature importance
feature_importance = best_rf.feature_importances_
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importance
})
importance_df = importance_df.sort_values('Importance', ascending=False).head(10)

# Plot feature importance
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df, palette='viridis')
plt.title('Top 10 Features - Random Forest Importance', fontsize=16)
plt.xlabel('Importance Score', fontsize=14)
plt.ylabel('Feature', fontsize=14)
plt.tick_params(axis='both', which='major', labelsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Simple Neural Network for Binary Classification
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix
import numpy as np

torch.manual_seed(1234)
np.random.seed(1234)

# Simple Neural Network Class
class SimpleConflictPredictor(nn.Module):
    def __init__(self, input_size):
        super(SimpleConflictPredictor, self).__init__()
        
        # Simple architecture: 2 hidden layers
        self.layer1 = nn.Linear(input_size, 64)
        self.layer2 = nn.Linear(64, 32)
        self.output = nn.Linear(32, 1)
        
        # Activation functions
        self.relu = nn.ReLU()          # ReLU for hidden layers
        self.sigmoid = nn.Sigmoid()    # Sigmoid for binary output
    
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.sigmoid(self.output(x))
        return x

# Prepare data (CPU only)
input_size = X_train_smote.shape[1]
print(f"Input features: {input_size}")

# Convert to PyTorch tensors (CPU)
X_train_tensor = torch.FloatTensor(X_train_smote)
y_train_tensor = torch.FloatTensor(y_train_smote.values.reshape(-1, 1))
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test.values.reshape(-1, 1))

# Create data loaders
batch_size = 128
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Initialize model
model = SimpleConflictPredictor(input_size)
print(f"Model architecture:")
print(model)

# Loss and optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy for binary classification
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)  # SGD optimizer

# Training function
def train_simple_model(model, train_loader, criterion, optimizer, num_epochs=50):
    model.train()
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(data)
            loss = criterion(outputs, target)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Statistics
            running_loss += loss.item()
            predicted = (outputs > 0.5).float()
            total += target.size(0)
            correct += (predicted == target).sum().item()
        
        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            accuracy = 100 * correct / total
            avg_loss = running_loss / len(train_loader)
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')

# Train the model
print("\nTraining Simple Neural Network with SGD...")
train_simple_model(model, train_loader, criterion, optimizer, num_epochs=50)

# Evaluation
def evaluate_simple_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test)
        test_preds = (test_outputs > 0.5).float()
        
        # Convert to numpy
        y_test_np = y_test.numpy().flatten()
        y_pred_np = test_preds.numpy().flatten()
        
        # Calculate metrics
        accuracy = (test_preds.flatten() == y_test.flatten()).float().mean().item()
        recall = recall_score(y_test_np, y_pred_np)
        precision = precision_score(y_test_np, y_pred_np)
        f1 = f1_score(y_test_np, y_pred_np)
        
        return accuracy, recall, precision, f1, y_pred_np

# Evaluate the model
accuracy, recall, precision, f1, y_pred_nn = evaluate_simple_model(model, X_test_tensor, y_test_tensor)

print("\n--- Simple Neural Network Results ---")
print(f"Accuracy: {accuracy:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"F1 Score: {f1:.4f}")

# Confusion Matrix
cm_nn = confusion_matrix(y_test, y_pred_nn)
print(f"\nConfusion Matrix:")
print(cm_nn)

# Compare with Random Forest
print(f"\n--- Model Comparison ---")
print(f"Random Forest Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"Neural Network Recall: {recall:.4f}")

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm_nn, annot=True, fmt='d', cmap='Reds',
            xticklabels=['No Conflict (0)', 'Conflict (1)'],
            yticklabels=['No Conflict (0)', 'Conflict (1)'])
plt.xlabel('Predicted Label', fontsize=14)
plt.ylabel('True Label', fontsize=14)
plt.title('Confusion Matrix - Simple Neural Network', fontsize=16)
plt.tight_layout()
plt.show()